# Crear la validación global

Este notebook crea **una única vez** la reserva de validación que comparten todas las
muestras de train (`create_samples.ipynb` la lee desde `VALIDACION_GLOBAL`). No se
sobrescribe: si ya existe, la ejecución se detiene.

Decisiones:

- **Filtro de países**: solo participan los países con `cantidad >= MIN_FILAS_PAIS` (1000)
  en `assets/countries_count.json`. Esos conteos coinciden con los anotados en
  `datasets/world-streetview-500k/about.txt` (110 países, 71 con 1000 o más imágenes).
  El umbral y la lista de países admitidos quedan guardados en `validation_config.json`.
- **Distribución desbalanceada**: la validación sigue la misma distribución que train.
  El cupo de cada país es proporcional a su cantidad en ese JSON (estratificado
  proporcional, por resto mayor).
- **Panoramas completos**: la unidad de muestreo es el `panoid`. De cada panorama elegido
  se toman todas sus imágenes válidas, porque train excluye el panorama entero y las
  vistas restantes se perderían. Por eso los cupos se asignan en panoramas: se aumenta
  la cantidad total de panoramas hasta que la cantidad de imágenes llega **al menos** a
  `TAMANO_MINIMO`.
- **Piso por país**: después del reparto proporcional, cada país con menos de
  `MIN_PANORAMAS_PAIS` (5) panoramas se sube a ese piso, sin quitarle panoramas a otros.
  Así ningún país queda con métricas basadas en 1 o 2 lugares; a cambio, el total supera
  `TAMANO_MINIMO` y los países chicos quedan algo sobrerrepresentados.
- Las filas con `image_id`/`panoid` vacíos no son elegibles. Se detiene si un panorama
  tiene varios países.
- La selección es reproducible con `SEMILLA`.

Salidas en `CARPETA_SALIDA`:

- `validation.parquet`: todas las columnas, con las imágenes sin decodificar.
- `validation_ids.parquet`: `image_id`, `panoid` y `country_code` de la reserva. Es el
  archivo para evitar el leakage al muestrear train: hay que excluir por `image_id` **y**
  por `panoid`.
- `validation_config.json`: parámetros, cupos por país, países admitidos y huella SHA-256 de los IDs.

In [ ]:
from pathlib import Path
import json
import hashlib
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

# Parámetros editables.
TAMANO_MINIMO = 5_000  # Mínimo de imágenes del reparto proporcional; se completa con panoramas enteros.
MIN_PANORAMAS_PAIS = 5  # Piso por país, agregado sobre el reparto proporcional.
MIN_FILAS_PAIS = 1000  # Umbral sobre los conteos de train (los de about.txt).
ARCHIVO_CONTEOS = "src/notebooks/preprocessing/assets/countries_count.json"
SEMILLA = 42
PATRON_ARCHIVOS = "**/train-*.parquet"
CARPETA_SALIDA = "datasets/samples/data/validation"  # Relativa a la raíz del proyecto, o absoluta.
TAMANO_LOTE = 128  # Limita la memoria al leer imágenes.

## Localizar el dataset y los conteos

`DATA_URL` se toma del entorno si ya está definida; de lo contrario se lee del `.env`.
Las rutas relativas se resuelven respecto de la carpeta del `.env`.

In [ ]:
def entero_valido(valor, nombre, minimo):
    if isinstance(valor, bool) or not isinstance(valor, (int, np.integer)) or valor < minimo:
        raise ValueError(f"{nombre} debe ser un entero >= {minimo}.")

for nombre in ("TAMANO_MINIMO", "MIN_PANORAMAS_PAIS", "MIN_FILAS_PAIS", "TAMANO_LOTE"):
    entero_valido(globals()[nombre], nombre, 1)
entero_valido(SEMILLA, "SEMILLA", 0)

env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")

raiz_proyecto = env_path.parent


def resolver(ruta):
    ruta = Path(ruta).expanduser()
    return (ruta if ruta.is_absolute() else raiz_proyecto / ruta).resolve()


ruta_datos = resolver(ruta_configurada)
if not ruta_datos.is_dir():
    raise NotADirectoryError(f"DATA_URL apunta a una carpeta inexistente: {ruta_datos}")
archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")

raiz_salida = resolver(CARPETA_SALIDA)
if raiz_salida == ruta_datos or ruta_datos in raiz_salida.parents:
    raise ValueError("CARPETA_SALIDA debe estar fuera de la carpeta del dataset de entrada.")
ruta_validacion = raiz_salida / "validation.parquet"
ruta_ids = raiz_salida / "validation_ids.parquet"
ruta_config = raiz_salida / "validation_config.json"
existentes = [p for p in (ruta_validacion, ruta_ids, ruta_config) if p.exists()]
if existentes:
    raise FileExistsError(f"La validación global ya existe: {existentes}. No se sobrescribe.")

print(f"Dataset: {ruta_datos}")
print(f"Archivos de entrada: {len(archivos):,}")

ruta_conteos = resolver(ARCHIVO_CONTEOS)


def sin_claves_repetidas(pares):
    # json.loads conserva en silencio la última clave repetida: detectarlo.
    claves = [clave.strip().upper() for clave, _ in pares]
    if len(claves) != len(set(claves)):
        raise ValueError("El JSON de conteos tiene codigos de pais repetidos.")
    return dict(pares)


# Objeto {codigo_pais: cantidad}; "NA" (Namibia) es una clave como cualquier otra.
conteos_json = json.loads(ruta_conteos.read_text(encoding="utf-8"), object_pairs_hook=sin_claves_repetidas)
if not isinstance(conteos_json, dict) or not conteos_json:
    raise ValueError("El JSON de conteos debe ser un objeto {codigo_pais: cantidad} no vacio.")
if any(not isinstance(pais, str) or not pais.strip() or isinstance(cantidad, bool)
       or not isinstance(cantidad, int) or cantidad < 0 for pais, cantidad in conteos_json.items()):
    raise ValueError("El JSON debe tener codigos presentes y cantidades enteras no negativas.")
conteos_train = pd.Series(
    {pais.strip().upper(): cantidad for pais, cantidad in conteos_json.items()}, dtype="int64"
).sort_index()
conteos_train.index.name = "pais"
paises_admitidos = conteos_train.index[conteos_train >= MIN_FILAS_PAIS]
if paises_admitidos.empty:
    raise ValueError("Ningun pais de los conteos alcanza MIN_FILAS_PAIS.")
print(f"Paises admitidos por los conteos (>= {MIN_FILAS_PAIS}): {len(paises_admitidos)}")
print(f"Imágenes de esos países según los conteos: {conteos_train[paises_admitidos].sum():,}")

## Leer países e identificadores

Se leen solo `country_code`, `image_id` y `panoid` de todos los archivos, y se mantiene
el índice global sobre los archivos ordenados para copiar luego las filas elegidas.
Se agrupan las imágenes por panorama y se comprueba que cada panorama tenga un único país.

In [ ]:
partes = []
inventario = []
esquema = None
for archivo in tqdm(archivos, desc="Leyendo países e identificadores"):
    with pq.ParquetFile(archivo) as parquet:
        if not {"country_code", "image_id", "panoid"}.issubset(parquet.schema_arrow.names):
            raise ValueError(f"Falta country_code, image_id o panoid en {archivo}")
        if esquema is None:
            esquema = parquet.schema_arrow
        elif not esquema.equals(parquet.schema_arrow, check_metadata=True):
            raise ValueError(f"El esquema difiere del primer archivo: {archivo}")
        partes.append(parquet.read(columns=["country_code", "image_id", "panoid"]).to_pandas())
        inventario.append({
            "archivo": str(archivo), "filas": parquet.metadata.num_rows,
            "bytes": archivo.stat().st_size, "mtime_ns": archivo.stat().st_mtime_ns,
        })

metadatos = pd.concat(partes, ignore_index=True)
del partes
paises = metadatos["country_code"].astype("string").str.strip().str.upper().replace("", pd.NA)
image_ids = metadatos["image_id"].astype("string").str.strip().replace("", pd.NA)
panoids = metadatos["panoid"].astype("string").str.strip().replace("", pd.NA)
del metadatos
ids_invalidos = image_ids.isna() | panoids.isna()
if image_ids.dropna().duplicated().any():
    raise ValueError("La fuente contiene image_id duplicados.")
paises_por_panoid = paises.loc[~ids_invalidos].groupby(panoids.loc[~ids_invalidos]).nunique(dropna=False)
if (paises_por_panoid > 1).any():
    raise ValueError("Hay panoramas asociados a varios países. Revisá la fuente.")

elegibles = ~ids_invalidos & paises.isin(paises_admitidos)
conteos_locales = paises.loc[elegibles].value_counts().sort_index()
print(f"Filas totales: {len(paises):,}; sin país: {paises.isna().sum():,}")
print(f"Identificadores inválidos: {ids_invalidos.sum():,}")
print(f"Filas elegibles (países admitidos, IDs válidos): {elegibles.sum():,}")
diferencias = (conteos_locales - conteos_train[paises_admitidos]).fillna(0)
if (diferencias != 0).any():
    print("Aviso: los conteos locales difieren del JSON en:")
    display(diferencias[diferencias != 0])

panoramas = pd.DataFrame({
    "panoid": panoids.loc[elegibles], "pais": paises.loc[elegibles],
}).groupby("panoid", sort=True).agg(pais=("pais", "first"), imagenes=("pais", "size"))
print(f"Panoramas elegibles: {len(panoramas):,}")
display(panoramas["imagenes"].value_counts().rename("panoramas").sort_index())

## Calcular cupos y seleccionar panoramas

Los pesos son los conteos de imágenes del JSON de los países admitidos. Para `N` panoramas
se reparten cupos por resto mayor: `N * cantidad_pais / total`, con desempates por
código de país. Dentro de cada país, los panoramas se ordenan una sola vez al azar
(con `SEMILLA`) y se toma el prefijo correspondiente al cupo. `N` arranca en
`ceil(TAMANO_MINIMO / imágenes medias por panorama)` y aumenta hasta que las imágenes
seleccionadas suman al menos `TAMANO_MINIMO`. Recién entonces se aplica el piso de
`MIN_PANORAMAS_PAIS` por país, que agrega panoramas por encima de ese mínimo.

In [ ]:
def cupos_resto_mayor(pesos, total):
    numeradores = pesos * total
    cupos = numeradores // pesos.sum()
    faltantes = total - int(cupos.sum())
    orden = (numeradores % pesos.sum()).sort_values(ascending=False, kind="stable")
    cupos.loc[orden.index[:faltantes]] += 1
    return cupos


pesos = conteos_train[paises_admitidos].sort_index().astype("int64")
rng = np.random.default_rng(SEMILLA)
# Orden aleatorio fijo por país: al aumentar N solo se extienden los prefijos.
orden_por_pais = {
    pais: rng.permutation(grupo.index.to_numpy())
    for pais, grupo in panoramas.groupby("pais", sort=True)
}
faltan = sorted(set(pesos.index) - set(orden_por_pais))
if faltan:
    raise ValueError(f"Países admitidos sin panoramas locales: {faltan}")

n_panoramas = int(np.ceil(TAMANO_MINIMO / panoramas["imagenes"].mean()))
while True:
    cupos = cupos_resto_mayor(pesos, n_panoramas)
    insuficientes = [p for p, c in cupos.items() if c > len(orden_por_pais[p])]
    if insuficientes:
        raise ValueError(f"No hay panoramas suficientes en: {insuficientes}")
    panoids_validacion = np.concatenate([orden_por_pais[p][:c] for p, c in cupos.items()])
    total_imagenes = int(panoramas.loc[panoids_validacion, "imagenes"].sum())
    if total_imagenes >= TAMANO_MINIMO:
        break
    n_panoramas += 1
print(f"Reparto proporcional: {len(panoids_validacion):,} panoramas; {total_imagenes:,} imágenes")

cupos_proporcionales = cupos.copy()
cupos = cupos.clip(lower=MIN_PANORAMAS_PAIS)
insuficientes = [p for p, c in cupos.items() if c > len(orden_por_pais[p])]
if insuficientes:
    raise ValueError(f"No hay panoramas suficientes para el piso en: {insuficientes}")
panoids_validacion = np.concatenate([orden_por_pais[p][:c] for p, c in cupos.items()])
total_imagenes = int(panoramas.loc[panoids_validacion, "imagenes"].sum())
subidos = cupos.index[cupos > cupos_proporcionales]
print(f"Países subidos al piso de {MIN_PANORAMAS_PAIS} panoramas: {len(subidos)} "
      f"(+{int((cupos - cupos_proporcionales).sum())} panoramas)")
print(f"Panoramas seleccionados: {len(panoids_validacion):,}; imágenes: {total_imagenes:,}")

## Revisar la selección

Se toman todas las imágenes elegibles de los panoramas elegidos. El resumen compara la
proporción de cada país en validación con su proporción en train (JSON de conteos).

In [ ]:
seleccionadas = elegibles & panoids.isin(panoids_validacion)
indices_validacion = np.flatnonzero(seleccionadas.to_numpy()).astype(np.int64)
ids_validacion = pd.DataFrame({
    "image_id": image_ids.iloc[indices_validacion].to_numpy(),
    "panoid": panoids.iloc[indices_validacion].to_numpy(),
    "country_code": paises.iloc[indices_validacion].to_numpy(),
})

assert len(ids_validacion) == total_imagenes >= TAMANO_MINIMO
assert ids_validacion["panoid"].nunique() == len(panoids_validacion)
assert not ids_validacion["image_id"].duplicated().any()
assert ids_validacion["country_code"].isin(paises_admitidos).all()
# Ninguna vista de los panoramas elegidos queda afuera (salvo IDs inválidos).
assert seleccionadas.sum() == (panoids.isin(panoids_validacion) & ~ids_invalidos).sum()

resumen = pd.DataFrame({
    "cantidad_train": pesos,
    "proporcion_train": pesos / pesos.sum(),
    "panoramas_proporcionales": cupos_proporcionales,
    "panoramas_validacion": cupos,
    "imagenes_validacion": ids_validacion["country_code"].value_counts().reindex(pesos.index, fill_value=0),
})
resumen["proporcion_validacion"] = resumen["imagenes_validacion"] / resumen["imagenes_validacion"].sum()
resumen.index.name = "country_code"
pd.set_option("display.max_rows", 200)
display(resumen.sort_values("cantidad_train", ascending=False))
print(f"Validación: {len(ids_validacion):,} imágenes, {len(panoids_validacion):,} panoramas, "
      f"{(resumen['imagenes_validacion'] > 0).sum()} países.")

## Guardar la validación

Se escribe `validation.parquet` por lotes en orden de origen, con el esquema original.
Después se verifica que los IDs escritos sean exactamente los seleccionados y recién
entonces se guardan `validation_ids.parquet` y `validation_config.json`.

In [ ]:
def guardar_filas(archivos, inventario, indices, esquema, destino, tamano_lote):
    for archivo, info in zip(archivos, inventario, strict=True):
        estado = archivo.stat()
        if estado.st_size != info["bytes"] or estado.st_mtime_ns != info["mtime_ns"]:
            raise RuntimeError(f"La fuente cambió desde el conteo: {archivo}. Reejecutá desde el inicio.")
    destino.parent.mkdir(parents=True, exist_ok=True)
    guardadas = 0
    inicio_archivo = 0
    with pq.ParquetWriter(destino, esquema, compression="snappy") as escritor:
        for archivo, info in tqdm(list(zip(archivos, inventario, strict=True)), desc="Guardando validación"):
            fin_archivo = inicio_archivo + info["filas"]
            seleccion = indices[np.searchsorted(indices, inicio_archivo):np.searchsorted(indices, fin_archivo)]
            if len(seleccion):
                with pq.ParquetFile(archivo) as parquet:
                    inicio_grupo = inicio_archivo
                    for grupo in range(parquet.num_row_groups):
                        fin_grupo = inicio_grupo + parquet.metadata.row_group(grupo).num_rows
                        if np.searchsorted(seleccion, inicio_grupo) < np.searchsorted(seleccion, fin_grupo):
                            inicio_lote = inicio_grupo
                            for lote in parquet.iter_batches(batch_size=tamano_lote, row_groups=[grupo]):
                                fin_lote = inicio_lote + lote.num_rows
                                globales = seleccion[np.searchsorted(seleccion, inicio_lote):np.searchsorted(seleccion, fin_lote)]
                                if len(globales):
                                    posiciones = globales - inicio_lote
                                    escritor.write_table(pa.Table.from_batches([lote]).take(pa.array(posiciones)))
                                    guardadas += len(posiciones)
                                inicio_lote = fin_lote
                        inicio_grupo = fin_grupo
            inicio_archivo = fin_archivo
    if guardadas != len(indices):
        raise RuntimeError(f"Filas guardadas: {guardadas}; se esperaban {len(indices)}.")
    return guardadas


guardadas = guardar_filas(archivos, inventario, indices_validacion, esquema, ruta_validacion, TAMANO_LOTE)
# Verificar lo escrito, no solo los índices seleccionados.
ids_guardados = pq.read_table(ruta_validacion, columns=["image_id", "panoid", "country_code"]).to_pandas()
for columna in ids_guardados:
    ids_guardados[columna] = ids_guardados[columna].astype("string").str.strip()
ids_guardados["country_code"] = ids_guardados["country_code"].str.upper()
if not ids_guardados.reset_index(drop=True).astype(object).equals(ids_validacion.astype(object)):
    raise RuntimeError("Los IDs escritos no coinciden con la selección.")

pq.write_table(pa.Table.from_pandas(ids_validacion, preserve_index=False), ruta_ids)
ids_ordenados = ids_validacion[["image_id", "panoid"]].sort_values(["image_id", "panoid"]).to_numpy().tolist()
huella_ids = hashlib.sha256(json.dumps(ids_ordenados, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
configuracion = {
    "tipo_muestra": "validacion_global",
    "tamano_minimo": TAMANO_MINIMO,
    "filas": len(ids_validacion),
    "panoramas": int(len(panoids_validacion)),
    "archivos_salida": {
        "validation": ruta_validacion.name,
        "ids": ruta_ids.name,
    },
    "sha256_ids": huella_ids,
    "columnas_exclusion_train": ["image_id", "panoid"],
    "min_filas_pais": MIN_FILAS_PAIS,
    "archivo_conteos": str(ruta_conteos),
    "fuente_conteos": "countries_count.json; coincide con datasets/world-streetview-500k/about.txt",
    "conteos_train": {pais: int(cantidad) for pais, cantidad in conteos_train.items()},
    "paises_admitidos": paises_admitidos.tolist(),
    "semilla": SEMILLA,
    "min_panoramas_pais": MIN_PANORAMAS_PAIS,
    "metodo": ("estratificado proporcional a cantidad_train (resto mayor), por panorama completo, "
               "con piso de min_panoramas_pais por país"),
    "unidad": "panoid (todas sus imágenes)",
    "cupos_panoramas_proporcionales": {pais: int(c) for pais, c in cupos_proporcionales.items()},
    "cupos_panoramas": {pais: int(c) for pais, c in cupos.items()},
    "imagenes_por_pais": {pais: int(n) for pais, n in resumen["imagenes_validacion"].items()},
    "ruta_datos": str(ruta_datos),
    "patron_archivos": PATRON_ARCHIVOS,
    "archivos_fuente": inventario,
    "filas_fuente": len(paises),
    "filas_sin_pais": int(paises.isna().sum()),
    "filas_ids_invalidos": int(ids_invalidos.sum()),
    "indices_globales": indices_validacion.tolist(),
}
ruta_config.write_text(json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Validación guardada: {guardadas:,} imágenes en {ruta_validacion}")
print(f"IDs: {ruta_ids}")
print(f"Configuración: {ruta_config}")